In [1]:
   import pandas as pd
   from pathlib import Path

   csv_path = next(Path("../data/raw").glob("*.csv"))
   df = pd.read_csv(csv_path)

   print(csv_path.name)
   print(df.shape)
   df.head()

PS_20174392719_1491204439457_log.csv
(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [3]:
df["isFraud"].value_counts()

isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [ ]:
df["isFraud"].value_counts(normalize=True) * 100

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [5]:
df.groupby("type")["isFraud"].agg(["count", "sum", "mean"])

,count,sum,mean
type,,,
CASH_IN,1399284,0,0.000000
CASH_OUT,2237500,4116,0.001840
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000
TRANSFER,532909,4097,0.007688


In [6]:
df.isna().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [7]:
df_model = df[df["type"].isin(["CASH_OUT", "TRANSFER"])].copy()
print(df_model.shape)
print(df_model["isFraud"].mean() * 100)

(2770409, 11)
0.2964544224336551


In [8]:
df_model.groupby("isFraud")["amount"].describe()

,count,mean,std,min,25%,50%,75%,max
isFraud,,,,,,,,
0,2762196.0,3.141155e+05,8.771441e+05,0.01,82908.2325,171034.46,305994.185,92445516.64
1,8213.0,1.467967e+06,2.404253e+06,0.00,127091.3300,441423.44,1517771.480,10000000.00


In [9]:
df["step"].agg(["min", "max"])

min      1
max    743
Name: step, dtype: int64

In [10]:
fraud = df_model[df_model["isFraud"] == 1]
legit = df_model[df_model["isFraud"] == 0]
print("Fraud with origin balance emptied:", (fraud["newbalanceOrig"] == 0).mean())
print("Legit with origin balance emptied:", (legit["newbalanceOrig"] == 0).mean())

Fraud with origin balance emptied: 0.9805186898818946
Legit with origin balance emptied: 0.9009509100729999


In [11]:
df["step"].agg(["min", "max"])

min      1
max    743
Name: step, dtype: int64

In [12]:
df["isFlaggedFraud"].value_counts()

isFlaggedFraud
0    6362604
1         16
Name: count, dtype: int64

In [13]:
fraud = df_model[df_model["isFraud"] == 1]
legit = df_model[df_model["isFraud"] == 0]

# Zero-amount transactions
print("Zero-amount rows:", (df_model["amount"] == 0).sum(), "of which fraud:", df_model.loc[df_model["amount"] == 0, "isFraud"].sum())

# Sender had money and ended with nothing
def drained(d):
    return ((d["oldbalanceOrg"] > 0) & (d["newbalanceOrig"] == 0)).mean()
print("Fraud: sender had money and was emptied:", drained(fraud))
print("Legit: sender had money and was emptied:", drained(legit))

# Destination balance unchanged at zero
def dest_zero(d):
    return ((d["oldbalanceDest"] == 0) & (d["newbalanceDest"] == 0)).mean()
print("Fraud: destination balance zero before and after:", dest_zero(fraud))
print("Legit: destination balance zero before and after:", dest_zero(legit))

Zero-amount rows: 16 of which fraud: 16
Fraud: sender had money and was emptied: 0.9755266041641301
Legit: sender had money and was emptied: 0.4272187781026401
Fraud: destination balance zero before and after: 0.49628637525873615
Legit: destination balance zero before and after: 0.0006176245277308345


In [14]:
df[df["isFlaggedFraud"] == 1]["isFraud"].value_counts()

isFraud
1    16
Name: count, dtype: int64

In [15]:
import pandas as pd
from pathlib import Path

csv_path = next(Path("../data/raw").glob("*.csv"))
df = pd.read_csv(csv_path)
df = df[df["type"].isin(["CASH_OUT", "TRANSFER"])].copy()
df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)
print(df.shape, df["isFraud"].mean() * 100)

(2770409, 12) 0.2964544224336551


In [16]:
cutoff = df["step"].quantile(0.8)
train = df[df["step"] <= cutoff]
test = df[df["step"] > cutoff]

for name, part in [("train", train), ("test", test)]:
    print(name, len(part), "fraud:", part["isFraud"].sum(), f"({part['isFraud'].mean()*100:.3f}%)")

train 2217905 fraud: 3955 (0.178%)
test 552504 fraud: 4258 (0.771%)


In [17]:
y_test = test["isFraud"]
print("Never-fraud accuracy:", (y_test == 0).mean())
print("Built-in rule flagged:", test["isFlaggedFraud"].sum(), "of", y_test.sum(), "fraud cases")

Never-fraud accuracy: 0.9922932684650246
Built-in rule flagged: 13 of 4258 fraud cases


In [18]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_score, recall_score

features = ["is_transfer", "amount", "oldbalanceOrg", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest"]

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(class_weight="balanced", max_iter=1000),
)
model.fit(train[features], train["isFraud"])

scores = model.predict_proba(test[features])[:, 1]
preds = scores >= 0.5

print("PR-AUC:", average_precision_score(y_test, scores))
print("Precision:", precision_score(y_test, preds))
print("Recall:", recall_score(y_test, preds))
print(confusion_matrix(y_test, preds))

PR-AUC: 0.7245509254447483
Precision: 0.11362443887366641
Recall: 0.9154532644434007
[[517838  30408]
 [   360   3898]]


In [19]:
import pandas as pd
from pathlib import Path

csv_path = next(Path("../data/raw").glob("*.csv"))
df = pd.read_csv(csv_path)
df = df[df["type"].isin(["CASH_OUT", "TRANSFER"])].copy()
df["is_transfer"] = (df["type"] == "TRANSFER").astype(int)

cutoff = df["step"].quantile(0.8)

In [20]:
df["error_orig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
df["error_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
df["orig_drained"] = ((df["oldbalanceOrg"] > 0) & (df["newbalanceOrig"] == 0)).astype(int)
df["dest_zero_both"] = ((df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)).astype(int)
df["amount_ratio"] = df["amount"] / (df["oldbalanceOrg"] + 1)
df["hour_of_day"] = df["step"] % 24

base_features = ["is_transfer", "amount", "oldbalanceOrg", "newbalanceOrig",
                 "oldbalanceDest", "newbalanceDest"]
new_features = ["error_orig", "error_dest", "orig_drained",
                "dest_zero_both", "amount_ratio", "hour_of_day"]

In [21]:
val_cutoff = df.loc[df["step"] <= cutoff, "step"].quantile(0.85)

train = df[df["step"] <= val_cutoff]
val = df[(df["step"] > val_cutoff) & (df["step"] <= cutoff)]
test = df[df["step"] > cutoff]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, len(part), "fraud:", part["isFraud"].sum())

train 1889936 fraud: 3471
val 327969 fraud: 484
test 552504 fraud: 4258


In [22]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

features = base_features + new_features

def fit_and_score(cols, label):
    m = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
    m.fit(train[cols], train["isFraud"])
    s = m.predict_proba(val[cols])[:, 1]
    print(label, "validation PR-AUC:", round(average_precision_score(val["isFraud"], s), 4))
    return m

m_base = fit_and_score(base_features, "Raw features only")
m_full = fit_and_score(features, "With engineered features")

Raw features only validation PR-AUC: 0.9042
With engineered features validation PR-AUC: 1.0


In [23]:
import numpy as np
from sklearn.metrics import precision_recall_curve

val_scores = m_full.predict_proba(val[features])[:, 1]
prec, rec, thr = precision_recall_curve(val["isFraud"], val_scores)

for target in [0.5, 0.7, 0.9]:
    ok = np.where(rec[:-1] >= target)[0]
    i = ok[-1]
    print(f"recall >= {target}: threshold {thr[i]:.3f}, precision {prec[i]:.3f}")

recall >= 0.5: threshold 0.999, precision 1.000
recall >= 0.7: threshold 0.999, precision 1.000
recall >= 0.9: threshold 0.999, precision 1.000


In [24]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

chosen = 0.5  # replace with your choice from cell 5
test_scores = m_full.predict_proba(test[features])[:, 1]
test_preds = test_scores >= chosen

print("Test PR-AUC:", average_precision_score(test["isFraud"], test_scores))
print("Precision:", precision_score(test["isFraud"], test_preds))
print("Recall:", recall_score(test["isFraud"], test_preds))
print(confusion_matrix(test["isFraud"], test_preds))

Test PR-AUC: 0.9999901999276696
Precision: 0.9925390533923991
Recall: 0.9997651479567872
[[548214     32]
 [     1   4257]]


In [25]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

def run(cols, label):
    m = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
    m.fit(train[cols], train["isFraud"])
    s = m.predict_proba(val[cols])[:, 1]
    print(f"{label:38s} val PR-AUC: {average_precision_score(val['isFraud'], s):.4f}")

run(base_features, "Raw features")
run(base_features + ["orig_drained", "dest_zero_both", "amount_ratio", "hour_of_day"],
    "Engineered, without error features")
run(base_features + new_features, "All features")

# Only information known BEFORE the transaction completes
pre_tx = ["is_transfer", "amount", "oldbalanceOrg", "oldbalanceDest",
          "amount_ratio", "hour_of_day"]
run(pre_tx, "Pre-transaction information only")

Raw features                           val PR-AUC: 0.9042
Engineered, without error features     val PR-AUC: 0.9979
All features                           val PR-AUC: 1.0000
Pre-transaction information only       val PR-AUC: 0.9937


In [26]:
print(train.groupby("isFraud")["amount_ratio"].describe())

full_drain = train["amount_ratio"] > 0.99
print(full_drain.groupby(train["isFraud"]).mean())

             count           mean            std       min       25%  \
isFraud                                                                
0        1886465.0  152954.297661  811549.982923  0.000003  5.057737   
1           3471.0    2748.069427   49640.969022  0.000000  0.999991   

                50%            75%          max  
isFraud                                          
0        670.453563  157316.940000  92445516.64  
1          0.999998       0.999999   1933920.80  
isFraud
0    0.900666
1    0.986747
Name: amount_ratio, dtype: float64


In [27]:
from sklearn.inspection import permutation_importance

final = HistGradientBoostingClassifier(class_weight="balanced", random_state=42)
final.fit(train[pre_tx], train["isFraud"])

result = permutation_importance(
    final, val[pre_tx], val["isFraud"],
    scoring="average_precision", n_repeats=3, random_state=42, n_jobs=-1,
)
imp = pd.Series(result.importances_mean, index=pre_tx).sort_values(ascending=False)
print(imp)

amount_ratio      0.991728
amount            0.006852
oldbalanceOrg     0.006085
oldbalanceDest    0.000113
is_transfer       0.000000
hour_of_day       0.000000
dtype: float64
